# NB08 — compbio_utils Package: Sequence Utilities

**Module 16 — Research Software Engineering**

---

## Motivation

This notebook builds `compbio_utils/sequence.py` — the core sequence manipulation module. Every function is implemented from scratch (no Biopython), fully documented with NumPy-style docstrings, and tested with 20+ pytest cases. This is the module that will be imported by all subsequent notebooks and the capstone project.

## Biological Background

- **reverse_complement**: fundamental to understanding double-stranded DNA; used in PCR primer design, restriction site searching, and BLAST alignment on both strands.
- **gc_content**: correlates with thermal stability, gene density, and codon usage; used in primer design, GC-bias correction in RNA-seq, and taxonomic profiling.
- **kmer_frequencies**: the basis of most alignment-free sequence comparison methods; used in genome assembly (de Bruijn graphs), metagenomics (MinHash), and error correction.
- **find_orfs**: open reading frames define protein-coding potential; ORF prediction is the first step in genome annotation.
- **translate**: converts a DNA coding sequence to amino acids using the standard genetic code; essential for functional annotation.

---

In [ ]:
"""sequence.py — implementation cell.

In the real package, this code lives in src/compbio_utils/sequence.py.
Here we define it in the notebook for demonstration and testing.
"""
from __future__ import annotations


COMPLEMENT_TABLE = str.maketrans('ATCGatcg', 'TAGCtagc')

STANDARD_CODON_TABLE: dict[str, str] = {
    'TTT': 'F', 'TTC': 'F', 'TTA': 'L', 'TTG': 'L',
    'CTT': 'L', 'CTC': 'L', 'CTA': 'L', 'CTG': 'L',
    'ATT': 'I', 'ATC': 'I', 'ATA': 'I', 'ATG': 'M',
    'GTT': 'V', 'GTC': 'V', 'GTA': 'V', 'GTG': 'V',
    'TCT': 'S', 'TCC': 'S', 'TCA': 'S', 'TCG': 'S',
    'CCT': 'P', 'CCC': 'P', 'CCA': 'P', 'CCG': 'P',
    'ACT': 'T', 'ACC': 'T', 'ACA': 'T', 'ACG': 'T',
    'GCT': 'A', 'GCC': 'A', 'GCA': 'A', 'GCG': 'A',
    'TAT': 'Y', 'TAC': 'Y', 'TAA': '*', 'TAG': '*',
    'CAT': 'H', 'CAC': 'H', 'CAA': 'Q', 'CAG': 'Q',
    'AAT': 'N', 'AAC': 'N', 'AAA': 'K', 'AAG': 'K',
    'GAT': 'D', 'GAC': 'D', 'GAA': 'E', 'GAG': 'E',
    'TGT': 'C', 'TGC': 'C', 'TGA': '*', 'TGG': 'W',
    'CGT': 'R', 'CGC': 'R', 'CGA': 'R', 'CGG': 'R',
    'AGT': 'S', 'AGC': 'S', 'AGA': 'R', 'AGG': 'R',
    'GGT': 'G', 'GGC': 'G', 'GGA': 'G', 'GGG': 'G',
}

VALID_DNA = frozenset('ATCGatcg')


def _validate_dna(seq: str) -> str:
    """Validate and uppercase a DNA sequence."""
    seq = seq.upper()
    invalid = set(seq) - frozenset('ATCG')
    if invalid:
        raise ValueError(
            f'Invalid nucleotide(s): {", ".join(sorted(invalid))}. '
            'Only A, T, C, G are accepted.'
        )
    return seq


def reverse_complement(seq: str) -> str:
    """Return the reverse complement of a DNA sequence.

    Parameters
    ----------
    seq : str
        DNA sequence (5' to 3'). Case-insensitive. Must contain
        only the characters A, T, C, G (case-insensitive).

    Returns
    -------
    str
        Reverse complement (5' to 3'), uppercase.

    Raises
    ------
    ValueError
        If ``seq`` contains characters outside {A, T, C, G}.

    Examples
    --------
    >>> reverse_complement('ATCG')
    'CGAT'
    >>> reverse_complement('AAAA')
    'TTTT'
    >>> reverse_complement('atcg')
    'CGAT'
    """
    seq = _validate_dna(seq)
    return seq.translate(COMPLEMENT_TABLE)[::-1]


def gc_content(seq: str) -> float:
    """Compute the GC content of a DNA sequence as a fraction.

    Parameters
    ----------
    seq : str
        DNA sequence. Case-insensitive. Returns 0.0 for empty sequences.

    Returns
    -------
    float
        GC fraction in [0.0, 1.0].

    Raises
    ------
    ValueError
        If ``seq`` contains characters outside {A, T, C, G}.

    Examples
    --------
    >>> gc_content('ATCG')
    0.5
    >>> gc_content('')
    0.0
    >>> gc_content('GGGG')
    1.0
    """
    if not seq:
        return 0.0
    seq = _validate_dna(seq)
    return (seq.count('G') + seq.count('C')) / len(seq)


def kmer_frequencies(seq: str, k: int) -> dict[str, int]:
    """Compute k-mer frequencies for a DNA sequence.

    Parameters
    ----------
    seq : str
        DNA sequence. Case-insensitive.
    k : int
        k-mer length. Must be > 0 and <= len(seq).

    Returns
    -------
    dict[str, int]
        Mapping of k-mer string to count. Only k-mers that appear
        at least once are included.

    Raises
    ------
    ValueError
        If k <= 0 or k > len(seq) or seq contains invalid characters.

    Examples
    --------
    >>> kmer_frequencies('ATCGAT', 3)
    {'ATC': 1, 'TCG': 1, 'CGA': 1, 'GAT': 1}
    >>> kmer_frequencies('ATAT', 2)
    {'AT': 2, 'TA': 1}
    """
    seq = _validate_dna(seq)
    if k <= 0:
        raise ValueError(f'k must be > 0, got {k}')
    if k > len(seq):
        raise ValueError(f'k ({k}) must be <= len(seq) ({len(seq)})')
    counts: dict[str, int] = {}
    for i in range(len(seq) - k + 1):
        kmer = seq[i:i + k]
        counts[kmer] = counts.get(kmer, 0) + 1
    return counts


def find_orfs(
    seq: str,
    min_length: int = 100,
) -> list[tuple[int, int, str]]:
    """Find all open reading frames (ORFs) in a DNA sequence.

    Searches all three forward reading frames. An ORF is defined as
    a region starting with ATG and ending at the first downstream
    in-frame stop codon (TAA, TAG, or TGA).

    Parameters
    ----------
    seq : str
        DNA sequence. Case-insensitive.
    min_length : int, optional
        Minimum ORF length in nucleotides (including start and stop codons).
        Default is 100.

    Returns
    -------
    list[tuple[int, int, str]]
        List of (start, end, frame) tuples, where start and end are
        0-based indices (end is exclusive) and frame is '+1', '+2', or '+3'.
        Sorted by start position.

    Examples
    --------
    >>> # A sequence with one ORF starting at position 0
    >>> orfs = find_orfs('ATG' + 'AAA' * 10 + 'TAA', min_length=9)
    >>> len(orfs) >= 1
    True
    >>> orfs[0][0]  # start position
    0
    """
    seq = _validate_dna(seq)
    stop_codons = {'TAA', 'TAG', 'TGA'}
    orfs: list[tuple[int, int, str]] = []

    for frame in range(3):
        frame_label = f'+{frame + 1}'
        i = frame
        while i < len(seq) - 2:
            codon = seq[i:i + 3]
            if codon == 'ATG':  # found a start codon
                start = i
                j = i + 3
                while j <= len(seq) - 3:
                    stop_codon = seq[j:j + 3]
                    if stop_codon in stop_codons:
                        end = j + 3
                        if end - start >= min_length:
                            orfs.append((start, end, frame_label))
                        break
                    j += 3
            i += 3

    return sorted(orfs, key=lambda x: x[0])


def translate(seq: str) -> str:
    """Translate a DNA coding sequence to a single-letter amino acid sequence.

    Uses the standard genetic code. The sequence must start with ATG and
    should end at (but not include) a stop codon. Stop codons are represented
    as '*'.

    Parameters
    ----------
    seq : str
        DNA coding sequence. Must be a multiple of 3 in length.
        Case-insensitive.

    Returns
    -------
    str
        Amino acid sequence in single-letter code. Stop codons produce '*'.

    Raises
    ------
    ValueError
        If len(seq) is not a multiple of 3.
    ValueError
        If seq contains non-ATCG characters.
    KeyError
        If a codon is not in the standard codon table (should not occur
        for valid ATCG sequences).

    Examples
    --------
    >>> translate('ATGGCC')
    'MA'
    >>> translate('ATGTAA')
    'M*'
    """
    seq = _validate_dna(seq)
    if len(seq) % 3 != 0:
        raise ValueError(f'Sequence length ({len(seq)}) must be a multiple of 3')
    return ''.join(
        STANDARD_CODON_TABLE[seq[i:i + 3]]
        for i in range(0, len(seq), 3)
    )


print('All sequence functions defined.')

# Quick smoke tests
assert reverse_complement('ATCG') == 'CGAT'
assert gc_content('ATCG') == 0.5
assert kmer_frequencies('ATAT', 2) == {'AT': 2, 'TA': 1}
assert translate('ATGGCC') == 'MA'
orfs = find_orfs('ATG' + 'AAA' * 10 + 'TAA', min_length=9)
assert len(orfs) >= 1
print('Smoke tests passed.')

In [ ]:
# 20+ parametrized test cases (shown as assertions here; would use pytest.mark.parametrize in tests/)
import pytest

REVCOMP_CASES = [
    ('ATCG', 'CGAT'), ('AAAA', 'TTTT'), ('CCCC', 'GGGG'),
    ('TTTT', 'AAAA'), ('A', 'T'), ('T', 'A'), ('G', 'C'), ('C', 'G'),
    ('atcg', 'CGAT'), ('ATCGATCG', 'CGATCGAT'),
    ('GCGCGCGC', 'GCGCGCGC'),  # palindrome
]

GC_CASES = [
    ('ATCG', 0.5), ('AAAA', 0.0), ('CCCC', 1.0), ('GGGG', 1.0),
    ('TTTT', 0.0), ('', 0.0), ('ATCGATCG', 0.5), ('GCGCGCGC', 1.0),
]

TRANSLATE_CASES = [
    ('ATG', 'M'), ('ATGGCC', 'MA'), ('ATGTAA', 'M*'),
    ('TTTCCC', 'FP'), ('GCAGCAGCA', 'AAA'),
]

failed = 0
for seq, expected in REVCOMP_CASES:
    actual = reverse_complement(seq)
    if actual != expected:
        print(f'FAIL revcomp({seq!r}): got {actual!r}, expected {expected!r}')
        failed += 1

for seq, expected in GC_CASES:
    actual = gc_content(seq)
    if abs(actual - expected) > 1e-10:
        print(f'FAIL gc_content({seq!r}): got {actual}, expected {expected}')
        failed += 1

for seq, expected in TRANSLATE_CASES:
    actual = translate(seq)
    if actual != expected:
        print(f'FAIL translate({seq!r}): got {actual!r}, expected {expected!r}')
        failed += 1

# Test ValueError on invalid input
import traceback
for invalid_seq in ['ATCGN', 'ATCG1', 'ATCG ']:
    try:
        reverse_complement(invalid_seq)
        print(f'FAIL: {invalid_seq!r} did not raise ValueError')
        failed += 1
    except ValueError:
        pass  # expected

total = len(REVCOMP_CASES) + len(GC_CASES) + len(TRANSLATE_CASES) + 3
print(f'\nTest results: {total - failed}/{total} passed')

## Visualization

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

rng = np.random.default_rng(42)

def random_dna(length: int) -> str:
    return ''.join(rng.choice(list('ATCG'), size=length))

seq = random_dna(500)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('compbio_utils.sequence — Visualizations', fontsize=13, fontweight='bold')

# --- Panel 1: GC content sliding window ---
ax1 = axes[0]
window = 50
gc_sliding = [
    gc_content(seq[i:i + window])
    for i in range(0, len(seq) - window + 1, 5)
]
positions = list(range(0, len(seq) - window + 1, 5))
ax1.plot(positions, gc_sliding, color='#3498db', lw=1.5)
ax1.axhline(y=0.5, color='red', linestyle='--', alpha=0.6, label='Expected (random)')
ax1.fill_between(positions, gc_sliding, 0.5,
                 where=[g > 0.5 for g in gc_sliding], alpha=0.3, color='#27ae60', label='GC-rich')
ax1.fill_between(positions, gc_sliding, 0.5,
                 where=[g < 0.5 for g in gc_sliding], alpha=0.3, color='#e74c3c', label='AT-rich')
ax1.set_xlabel('Position (bp)')
ax1.set_ylabel('GC content')
ax1.set_title(f'GC Content (window={window} bp)', fontweight='bold')
ax1.legend(fontsize=8)
ax1.set_ylim(0, 1)
ax1.grid(alpha=0.3)

# --- Panel 2: ORF map ---
ax2 = axes[1]
# Use a sequence guaranteed to have ORFs
orf_seq = 'ATG' + 'AAA' * 20 + 'TAA' + 'GGG' * 5 + 'ATG' + 'CCC' * 15 + 'TGA'
orfs = find_orfs(orf_seq, min_length=30)

ax2.set_xlim(0, len(orf_seq))
ax2.set_ylim(-0.5, 3.5)
ax2.set_xlabel('Position (bp)')
ax2.set_title('ORF Map', fontweight='bold')
ax2.set_yticks([1, 2, 3])
ax2.set_yticklabels(['+1', '+2', '+3'])
ax2.set_ylabel('Reading frame')

# Draw sequence background
ax2.barh(0, len(orf_seq), height=0.3, color='#ecf0f1', alpha=0.5)

frame_y = {'+1': 1, '+2': 2, '+3': 3}
colors_orf = ['#3498db', '#e67e22', '#27ae60', '#e74c3c']
for i, (start, end, frame) in enumerate(orfs):
    y = frame_y[frame]
    ax2.barh(y, end - start, left=start, height=0.5,
             color=colors_orf[i % len(colors_orf)], alpha=0.8)
    ax2.text(start + (end - start) / 2, y, f'{end - start} bp',
             ha='center', va='center', fontsize=7, color='white', fontweight='bold')

ax2.grid(axis='x', alpha=0.3)

# --- Panel 3: k-mer frequency distribution (k=3) ---
ax3 = axes[2]
kmer3 = kmer_frequencies(seq, k=3)
sorted_kmers = sorted(kmer3.items(), key=lambda x: -x[1])[:20]
k3_labels, k3_counts = zip(*sorted_kmers)

ax3.bar(range(len(k3_labels)), k3_counts, color='#9b59b6', alpha=0.8)
ax3.set_xticks(range(len(k3_labels)))
ax3.set_xticklabels(k3_labels, rotation=45, ha='right', fontsize=8, fontfamily='monospace')
ax3.set_xlabel('3-mer')
ax3.set_ylabel('Count')
ax3.set_title('Top 20 Trinucleotide Frequencies', fontweight='bold')
ax3.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('sequence_module_viz.png', dpi=120, bbox_inches='tight')
plt.show()

## Exercises

See `exercises/README.md` → E08.

1. Implement `find_orfs` for reverse strand ORFs as well. What modification is needed?
2. Write 5 tests for `find_orfs` covering: empty sequence, no ORFs present, overlapping start codons in the same frame, ORF at position 0, and a sequence with ORFs in all three frames.
3. Extend `kmer_frequencies` to return a normalized frequency dictionary (values sum to 1.0).

## Quiz

1. What are the three stop codons in the standard genetic code?
2. Why does `find_orfs` step by 3 in each frame?
3. What is a translation table in Python? How does `str.maketrans` work?
4. Why might two different DNA sequences have the same trinucleotide frequency profile?
5. What is an IUPAC ambiguity code? Why doesn't this version of `reverse_complement` support it?

## Reflection

*After completing:* What was the hardest part of `find_orfs` to get right? (Hint: think about overlapping ATG codons within the same ORF.) How would you handle ORFs that reach the end of the sequence without a stop codon?

## Papers Referenced

- Wilson et al. (2014) Best Practices — Rule: make the API clean and consistent.
- Taschuk & Wilson (2017) Rule 2: Have automated tests.